## How to use OpenAI models through Azure 

This is the simplest way to use OpenAI models that is somewhat documented:

* [Azure OpenAI Service documentation](https://learn.microsoft.com/en-gb/azure/ai-services/openai/)
* [Work with chat completions models](https://learn.microsoft.com/en-gb/azure/ai-services/openai/how-to/chatgpt)
* [OpenAI API Reference](https://platform.openai.com/docs/api-reference/introduction)
* [OpenAI API Reference: Libraries](https://platform.openai.com/docs/libraries?language=python)


#### I. Load in the configuration instead of hard-wiring API keys

* This step is essential or otherwise you **expose** API keys in GIT commits. 
* Make sure that configuration itself is not committed into GIT repository.
* Make sure that configparser indeed parsed the configuration you wanted.

In [1]:
from configparser import ConfigParser

CONFIG_DIR = "../model_configurations" 

config = ConfigParser()
status = config.read(f'{CONFIG_DIR}/azure_gpt-35-turbo.ini') 
assert status == [f'{CONFIG_DIR}/azure_gpt-35-turbo.ini']

#### II. Create OpenAI client
* Use helper function. It checks the configuration structure and assembles right fields. 

In [2]:
from llm_library.openai import configure_azure_client

client = configure_azure_client(config)

#### III. Use chat completion API to query model

* Client alone is not sufficient to interact with OpenAI.
* You need to specify model. This can be either model name or your deployment name.
* Experiment to determine what is right.
* It is polite to close the connection after all interactions but leaving it open does not cost extra money.  
  

In [3]:
response = client.chat.completions.create(
    model = config['azure-configuration']['deployment_name'],
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Does Azure OpenAI support customer managed keys?"},
        {"role": "assistant", "content": "Yes, customer managed keys are supported by Azure OpenAI."},
        {"role": "user", "content": "Do other Azure AI services support this too?"}
    ]
)
print(response.choices[0].message.content)
client.close()

Yes, other Azure AI services also support customer managed keys for enhanced security and control over data encryption and access.


## How to use OpenAI models through Azure with outlines library

The Outlines library adds nice support to structured output generation which is cumbersome to use directly through OpenAI API:
* [OpenAI API Reference: Structured Outputs](https://platform.openai.com/docs/guides/structured-outputs?api-mode=responses)
* [Github page for outlines](https://github.com/dottxt-ai/outlines)

#### I. Load in the configuration instead of hard-wiring API keys

* This step is essential or otherwise you **expose** API keys in GIT commits. 
* Make sure that configuration itself is not committed into GIT repository.
* Make sure that configparser indeed parsed the configuration you wanted.

In [4]:
from configparser import ConfigParser

CONFIG_DIR = "../model_configurations" 

config = ConfigParser()
status = config.read(f'{CONFIG_DIR}/azure_gpt-35-turbo.ini') 
assert status == [f'{CONFIG_DIR}/azure_gpt-35-turbo.ini']

#### II. Create OpenAI client
* Use helper function. It checks the configuration structure and assembles right fields. 

In [5]:
from llm_library.openai import configure_outlines_azure_client

client = configure_outlines_azure_client(config)

#### III. Use simplified chat completion API to query the model

* The default way cuts all the crap and let you work wihtout thinking of irrelevant details.
* You can access most of the OpenAI arguments like `max_tokens` and `temperature`.
* However, you loose direct access to `assistant` prompts that is one way to do fewshot learning.
* On the other hand, you have direct support to fewshot learning through propt templates ([Prompt templating](https://dottxt-ai.github.io/outlines/latest/reference/prompting/))  
 

In [6]:
?client.__call__

Signature:
client.__call__(
    prompt: Union[str, List[str]],
    max_tokens: Optional[int] = None,
    stop_at: Union[str, List[str], NoneType] = None,
    *,
    system_prompt: Optional[str] = None,
    temperature: Optional[float] = None,
    samples: Optional[int] = None,
) -> numpy.ndarray
Docstring:
Call the OpenAI API to generate text.

Parameters
----------
prompt
    A string or list of strings that will be used to prompt the model
max_tokens
    The maximum number of tokens to generate
stop_at
    A string or array of strings which, such that the generation stops
    when they are generated.
system_prompt
    The content of the system message that precedes the user's prompt.
temperature
    The value of the temperature used to sample tokens
samples
    The number of completions to generate for each prompt
stop_at
    Up to 4 words where the API will stop the completion.
File:      ~/Library/miniforge3/envs/chat-gpt/lib/python3.10/site-packages/outlines/models/openai.py
Type:

In [7]:
client("What is time?")

'Time is a concept that helps us measure and understand the progression of events and changes in our surroundings. It is a fundamental aspect of our reality that allows us to organize and make sense of our experiences. Time can be measured in various ways, such as seconds, minutes, hours, days, months, and years. It plays a crucial role in all aspects of our lives, influencing our daily routines, relationships, and overall perception of the world.'

#### IV. Use Simplified API for structured output generation
* Specify the expected output structure with pydantic
* Use generator to force the desired output.
* There are several output options: text, choice, json, regex, ...  


In [8]:
import outlines
from pydantic import BaseModel, ConfigDict

class User(BaseModel):
    name: str
    age: int
    city: str

In [9]:
[x for x in dir(outlines.generate) if x[0] != '_']

['SequenceGenerator',
 'api',
 'cfg',
 'choice',
 'format',
 'fsm',
 'generator',
 'json',
 'regex',
 'text']

In [10]:
?outlines.generate.json

Signature:
outlines.generate.json(
    model,
    schema_object: Union[str, object, Callable],
    sampler: outlines.samplers.Sampler = <outlines.samplers.MultinomialSampler object at 0x12852ba90>,
    whitespace_pattern: Optional[str] = None,
) -> outlines.generate.api.SequenceGeneratorAdapter
Docstring:
   Generate structured JSON data with a `Transformer` model based on a specified JSON Schema.

   Parameters
   ----------
   model:
       An instance of `Transformer` that represents a model from the
       `transformers` library.
   schema_object:
       The JSON Schema to generate data for. Can be a JSON string, a Pydantic model, or a callable
       that returns a JSON schema.
   sampler:
       The sampling algorithm to use to generate token ids from the logits
       distribution.
   whitespace_pattern
       Pattern to use for JSON syntactic whitespace (doesn't impact string literals)
       Example: allow only a single space or newline with `whitespace_pattern=r"[
]?"`

   Re

In [11]:
?outlines.generate.text

Signature:
outlines.generate.text(
    model,
    sampler: outlines.samplers.Sampler = <outlines.samplers.MultinomialSampler object at 0x15266a7a0>,
) -> outlines.generate.api.SequenceGeneratorAdapter
Docstring:
Generate text with a `Transformer` model.

Note
----
Python 3.11 allows dispatching on Union types and
this should greatly simplify the code.

Arguments
---------
model:
    An instance of `Transformer` that represents a model from the
    `transformers` library.
sampler:
    The sampling algorithm to use to generate token ids from the logits
    distribution.

Returns
-------
A `SequenceGeneratorAdapter` instance that generates text.
File:      ~/Library/miniforge3/envs/chat-gpt/lib/python3.10/site-packages/outlines/generate/text.py
Type:      function

In [12]:
generator = outlines.generate.json(client, User)

In [13]:
?generator.__call__

Signature:
generator.__call__(
    prompt: Union[str, List[str]],
    max_tokens: Optional[int] = None,
    stop_at: Union[str, List[str], NoneType] = None,
    *,
    system_prompt: Optional[str] = None,
    temperature: Optional[float] = None,
    samples: Optional[int] = None,
) -> numpy.ndarray
Docstring:
Call the OpenAI API to generate text.

Parameters
----------
prompt
    A string or list of strings that will be used to prompt the model
max_tokens
    The maximum number of tokens to generate
stop_at
    A string or array of strings which, such that the generation stops
    when they are generated.
system_prompt
    The content of the system message that precedes the user's prompt.
temperature
    The value of the temperature used to sample tokens
samples
    The number of completions to generate for each prompt
stop_at
    Up to 4 words where the API will stop the completion.
File:      ~/Library/miniforge3/envs/chat-gpt/lib/python3.10/site-packages/outlines/models/openai.py
Ty

In [14]:
# Does not work as OpenAI gpt-35-xxx does not support structured output generation
# generator("Generate a JSON object with name, age, and city.")